Classificador do arquivo CSV de notícias. 


In [ ]:
import os

def download_and_save(url, root_drive):
    """
    Faz o download de um arquivo do GitHub e o salva no Google Drive.

    Parâmetros:
    url (str): URL do arquivo no GitHub.
    root_drive (str): Caminho onde o arquivo será salvo no Google Drive.
    """
    file_name = os.path.basename(url)  # Extrai o nome do arquivo da URL

    # Faz o download do arquivo
    !wget {url} -O {file_name}

    # Move para a pasta destinada 
    !mv {file_name} {root_drive}

    print(f"✅ Arquivo '{file_name}' salvo em '{root_drive}'.")

# Caminho para donwload do arquivo CSV do GitHub e onde ele será salvo no computador
url_github = "https://raw.githubusercontent.com/ronaldandrade/ebook/07f1cccb41b70488fd000a153139aa814183631a/dados_exemplo/noticias.csv"
root_drive = "/home/ronald/Projetos/noticias/noticias.csv"

download_and_save(url_github, root_drive)



--2026-03-11 19:23:29--  https://raw.githubusercontent.com/ronaldandrade/ebook/07f1cccb41b70488fd000a153139aa814183631a/dados_exemplo/noticias.csv
Resolvendo raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8002::154, 2606:50c0:8001::154, 2606:50c0:8000::154, ...
Conectando-se a raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8002::154|:443... conectado.
A requisição HTTP foi enviada, aguardando resposta... 200 OK
Tamanho: 2388855 (2,3M) [text/plain]
Salvando em: “noticias.csv”

noticias.csv        100%[===================>]   2,28M  11,7MB/s    em 0,2s    

2026-03-11 19:23:29 (11,7 MB/s) - “noticias.csv” salvo [2388855/2388855]

mv: 'noticias.csv' e '/home/ronald/Projetos/noticias/noticias.csv' são o mesmo arquivo
✅ Arquivo 'noticias.csv' salvo em '/home/ronald/Projetos/noticias/noticias.csv'.


In [4]:
import pandas as pd
caminho = 'noticias.csv'  # Arquivo CSV com as notícias
df = pd.read_csv(caminho)  # Fazer leitura do arquivo CSV usando pandas
titulos = df['titulo'].tolist()  # Pega todos os títulos
print("Primeiros 5 títulos:", titulos[:5])  # Mostra os 5 primeiros pra conferir


Primeiros 5 títulos: ['Inflação argentina fica em 2,4% em fevereiro e cai para 66,9% em 12 meses', 'Lula diz que vai enviar ao Congresso isenção de IR para quem ganha até R$ 5 mil na próxima terça', 'Argentina e EUA lideram as quedas entre as principais bolsas globais, enquanto Europa e países emergentes sobem', 'Produção de veículos em fevereiro retoma patamares pré-pandemia após seis anos', 'Preço do café já subiu quase 40% no mundo e alta deve durar pelo menos quatro anos, diz ONU']


In [ ]:
amostra_titulos = titulos[:200]  # Só os 200 primeiros têm rótulos
rotulos = [1 if s == 'Positivo' else 0 for s in df['sentimento'][:200]]  # Converter a coluna 'sentimento'
print("Primeiros 5 rótulos:", rotulos[:5])  # Mostra os 5 primeiros rótulos
print("Total de positivos:", sum(rotulos), "Total de negativos:", len(rotulos) - sum(rotulos))

Primeiros 5 rótulos: [1, 1, 0, 1, 0]
Total de positivos: 108 Total de negativos: 92


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
stopwords_pt = ['e', 'em', 'de', 'para', 'com', 'que', 'do', 'da', 'no', 'na', 'o', 'a', 'as', 'os'] # Lista de stopwords em português
vetorizador = TfidfVectorizer(max_features=1000, stop_words=stopwords_pt, lowercase=True)
X = vetorizador.fit_transform(amostra_titulos).toarray()  # Converte os 200 títulos em números
print("Formato dos dados:", X.shape)  # Mostra quantas linhas e colunas temos


Formato dos dados: (200, 1000)


In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, rotulos, test_size=0.2, random_state=42)
print("Tamanho do treino:", len(y_train), "Tamanho do teste:", len(y_test))


Tamanho do treino: 160 Tamanho do teste: 40


In [9]:
from sklearn.svm import SVC
modelo = SVC(kernel='linear', class_weight='balanced')  # 'balanced' ajuda porque temos mais parâmetros de um tipo do que do outro.
modelo.fit(X_train, y_train)  # Treina o modelo com os dados de treino


SVC(class_weight='balanced', kernel='linear')

In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score
y_pred = modelo.predict(X_test)  # Faz previsões nos dados de teste
print("Acurácia no teste:", accuracy_score(y_test, y_pred))  # % de acertos
print("Precisão:", precision_score(y_test, y_pred))  # % de positivos previstos que são certos
print("Recall:", recall_score(y_test, y_pred))  # % de positivos reais que foram encontrados


Acurácia no teste: 0.6
Precisão: 0.5833333333333334
Recall: 0.7


In [ ]:
novos_titulos = ["Dólar cai e inflação diminui", "EUA tem maior número de deportaçoes nos últimos anos"] # Exemplos de possíveis notícias para testar o modelo
novos_X = vetorizador.transform(novos_titulos).toarray()  # Converter as novas notícias em números (vetorização)
predicoes = modelo.predict(novos_X)  # Fazer as previsões para as novas notícias
print("Previsões novas:", ["Positivo" if p == 1 else "Negativo" for p in predicoes])


Previsões novas: ['Positivo', 'Negativo']


In [ ]:
X_todas = vetorizador.transform(titulos).toarray()  # Converte todas as notícias em números
predicoes_todas = modelo.predict(X_todas)  # Prevê o sentimento de todas as notícias
df['sentimento'] = ['Positivo' if p == 1 else 'Negativo' for p in predicoes_todas]  # Atualiza a coluna 'sentimento' com as previsões
df.to_excel('noticias_classificadas.xlsx', index=False) # Salva a planilha com as notícias classificadas
print("Todas as 845 notícias classificadas e salvas em 'noticias_classificadas.xlsx'!")

Todas as 845 notícias classificadas e salvas em 'noticias_classificadas.xlsx'!
